# Tuần 4 — Đánh giá & So sánh 3 hệ (No-RAG / Standard RAG / Self-RAG-inspired)

Notebook này **không retrieval/embedding gì thêm** — chỉ load kết quả đã có từ Notebook 2 (`standard_rag_results.jsonl`) và Notebook 3 (`self_rag_results.jsonl`), sinh thêm hệ **No-RAG** (baseline dưới cùng, không retrieval), rồi chấm điểm hậu kiểm thống nhất cho cả 3 hệ bằng LLM-judge.

**Yêu cầu trước khi chạy**: đã chạy `01_retrieval_baseline.ipynb`, `02_generator_baseline.ipynb`, `03_self_rag_pipeline.ipynb` (không cần chạy hết 250 câu — notebook này tự tính trên phần giao nhau các câu đã xong, và báo rõ cỡ mẫu thực tế).

**3 chỉ số hậu kiểm dùng chung cho cả 3 hệ** (khác với ISSUP/ISUSE nội bộ của Notebook 3 vốn chỉ tính cho Self-RAG):

- **Correctness** (mới — so với `answer` gold trong `train.json`): CORRECT / PARTIALLY_CORRECT / INCORRECT.
- **Support** (ISSUP, chấm lại cho No-RAG/Standard RAG bằng đúng prompt của Notebook 3; **tái dùng** kết quả có sẵn cho Self-RAG để tiết kiệm API call): FULLY_SUPPORTED / PARTIALLY_SUPPORTED / NOT_SUPPORTED / NOT_APPLICABLE (No-RAG luôn NOT_APPLICABLE vì không có evidence).
- **Usefulness** (ISUSE, chấm lại cho No-RAG/Standard RAG; tái dùng cho Self-RAG): USEFUL / PARTIALLY_USEFUL / NOT_USEFUL.

Output: `final_comparison_table.csv` (bảng tổng hợp, dùng thẳng cho báo cáo) + `evaluation_details.jsonl` (chi tiết từng câu, dùng cho case study).

## 0. Cấu hình môi trường + API key

Giống Notebook 2/3. Không cần cài `sentence-transformers`/`faiss-cpu` ở đây vì không retrieval lại — chỉ dựng lại text điều luật từ `aid` có sẵn để chấm ISSUP cho Standard RAG.

In [ ]:
!pip install -q groq pandas

In [ ]:
import os

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA_ROOT = "/content/drive/MyDrive/NLP-CS2308.CH203-data"  # sua neu ban dat ten khac
    DATA_DIR = os.path.join(DRIVE_DATA_ROOT, "VLQA")
    ARTIFACT_DIR = os.path.join(DRIVE_DATA_ROOT, "artifacts")

    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    DATA_DIR = os.path.join(REPO_DIR, "final", "dataset", "VLQA")
    ARTIFACT_DIR = os.path.join(REPO_DIR, "final", "artifacts")
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    import getpass
    GROQ_API_KEY = getpass.getpass("Nhap GROQ_API_KEY: ")

print("DATA_DIR:", DATA_DIR, "| exists:", os.path.isdir(DATA_DIR))
print("ARTIFACT_DIR:", ARTIFACT_DIR, "| exists:", os.path.isdir(ARTIFACT_DIR))
print("GROQ_API_KEY loaded:", bool(GROQ_API_KEY))

## 1. Load dev set + kết quả Standard RAG / Self-RAG đã có

Vì free tier hay bị rate-limit (xem `PIPELINE.md` §6.4), **không giả định cả 250 câu đã chạy xong** — chỉ lấy phần giao nhau các `qid` đã có kết quả thành công ở cả 2 hệ, và báo rõ cỡ mẫu thực tế dùng để so sánh.

In [ ]:
import json

SPLIT_PATH = os.path.join(ARTIFACT_DIR, "dev_split_qids.json")
if not os.path.exists(SPLIT_PATH):
    raise FileNotFoundError(f"Khong tim thay {SPLIT_PATH} - hay chay 01_retrieval_baseline.ipynb truoc")
with open(SPLIT_PATH, encoding="utf-8") as f:
    dev_qids = set(json.load(f))

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_full = json.load(f)
qid_to_example = {ex["qid"]: ex for ex in train_full if ex["qid"] in dev_qids}


def load_jsonl_by_qid(path):
    result = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                if rec.get("generated_answer"):
                    result[rec["qid"]] = rec
    return result


standard_by_qid = load_jsonl_by_qid(os.path.join(ARTIFACT_DIR, "standard_rag_results.jsonl"))
self_by_qid = load_jsonl_by_qid(os.path.join(ARTIFACT_DIR, "self_rag_results.jsonl"))

print(f"Dev set: {len(dev_qids)} cau")
print(f"Standard RAG da co ket qua: {len(standard_by_qid)} cau")
print(f"Self-RAG-inspired da co ket qua: {len(self_by_qid)} cau")

common_qids = sorted(set(standard_by_qid) & set(self_by_qid) & dev_qids)
print(f"So cau co du ket qua ca 2 he (dung de sinh No-RAG + so sanh): {len(common_qids)}")

## 2. Dựng lại text điều luật từ `aid` (phục vụ chấm ISSUP cho Standard RAG)

`standard_rag_results.jsonl` chỉ lưu `retrieved_aids`, không lưu nguyên văn — cần tra lại `legal_corpus.json` để có text làm evidence khi chấm `[ISSUP]`.

In [ ]:
with open(os.path.join(DATA_DIR, "legal_corpus.json"), encoding="utf-8") as f:
    corpus = json.load(f)

aid_to_article = {}
for doc in corpus:
    for art in doc["content"]:
        aid_to_article[art["aid"]] = {"law_id": doc["law_id"], "text": art["content_Article"]}


def aids_to_chunks(aids):
    return [{"law_id": aid_to_article[aid]["law_id"], "text": aid_to_article[aid]["text"]} for aid in aids if aid in aid_to_article]


def build_context(chunks_list):
    return "\n\n".join(f"[{i}] (Van ban: {c['law_id']})\n{c['text']}" for i, c in enumerate(chunks_list, start=1))


print(f"Da load {len(aid_to_article)} dieu luat de tra cuu")

## 3. Groq client + dò model khả dụng

Giống Notebook 2/3 — không hardcode model, loại các model không phải chat/instruct, bỏ `groq/compound*` (đã xác nhận bị khóa cấp project trên tài khoản, xem `PIPELINE.md` §6.4).

In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

available_models = sorted(m.id for m in client.models.list().data)
print("Model kha dung tren tai khoan nay:")
for m in available_models:
    print(" -", m)

NON_CHAT_PREFIXES = ("whisper", "canopylabs/", "meta-llama/llama-prompt-guard")
chat_models = [m for m in available_models if not m.startswith(NON_CHAT_PREFIXES)]

CANDIDATE_MODELS = [
    "qwen/qwen3.6-27b",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
]
GENERATOR_MODEL = next((m for m in CANDIDATE_MODELS if m in chat_models), None) or (chat_models[0] if chat_models else None)
if GENERATOR_MODEL is None:
    raise RuntimeError("Khong tim thay model chat/instruct nao kha dung tren tai khoan nay.")
print("\nDang dung GENERATOR_MODEL =", GENERATOR_MODEL)

## 4. Hàm gọi LLM dùng chung (`chat()` rate-limit-aware) + `extract_json()`

Y nguyên cơ chế từ Notebook 3 (Retry-After ngắn thì chờ, dài thì tự chuyển model).

In [ ]:
import re
import time
from groq import RateLimitError, APIStatusError

LONG_WAIT_THRESHOLD = 30
exhausted_models = set()


def retry_after_seconds(exc, fallback):
    response = getattr(exc, "response", None)
    header = response.headers.get("retry-after") if response is not None else None
    if header is None:
        return fallback
    try:
        return float(header)
    except ValueError:
        return fallback


def model_queue():
    queue = [m for m in CANDIDATE_MODELS if m in chat_models and m not in exhausted_models]
    return queue or ([GENERATOR_MODEL] if GENERATOR_MODEL not in exhausted_models else [])


def chat(prompt, temperature=0.2, max_retries=4):
    queue = model_queue()
    if not queue:
        print("Tat ca model kha dung deu dang bi khoa dai han/loi - dung lai, thu lai sau.")
        return ""

    for model in queue:
        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                )
                return (response.choices[0].message.content or "").strip()
            except RateLimitError as e:
                wait = retry_after_seconds(e, fallback=2 ** attempt)
                if wait > LONG_WAIT_THRESHOLD:
                    print(f"Model {model} bi khoa dai han (Retry-After {wait:.0f}s) -> chuyen model tiep theo")
                    exhausted_models.add(model)
                    break
                print(f"Rate limit model {model} (lan {attempt + 1}/{max_retries}) -> cho {wait:.1f}s")
                time.sleep(wait)
            except APIStatusError as e:
                if e.status_code >= 500:
                    time.sleep(2 ** attempt)
                else:
                    print(f"Model {model} loi khong the retry (status {e.status_code}): {e} -> chuyen model tiep theo")
                    exhausted_models.add(model)
                    break
            except Exception as e:
                print(f"Loi khac (lan {attempt + 1}/{max_retries}): {e}")
                time.sleep(2 ** attempt)

    print("Tat ca model trong queue deu that bai (khoa dai han hoac loi) o lan chay nay.")
    return ""


def extract_json(text):
    if not text:
        return None
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    match = re.search(r"(\{.*\}|\[.*\])", cleaned, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            return None
    return None

## 5. Hệ No-RAG: sinh câu trả lời không cần retrieval

Baseline dưới cùng — chỉ hỏi LLM bằng kiến thức chung, không có điều luật nào được cấp. Cùng prompt `NO_CONTEXT_PROMPT` mà Notebook 3 dùng khi `[Retrieve] = NO_RETRIEVE`, để nhất quán phương pháp.

In [ ]:
NO_CONTEXT_PROMPT = """Ban la tro ly tu van phap luat Viet Nam. Khong co dieu luat cu the nao duoc cung cap cho cau hoi nay. Hay tra loi dua tren kien thuc chung, va noi ro day khong phai trich dan tu mot dieu luat cu the.

Cau hoi: {question}

Tra loi (tieng Viet, ngan gon):"""


def generate_norag_answer(question):
    return chat(NO_CONTEXT_PROMPT.format(question=question), temperature=0.2)


MAX_QUESTIONS = None
SLEEP_BETWEEN_CALLS = 2
NORAG_PATH = os.path.join(ARTIFACT_DIR, "no_rag_results.jsonl")

existing_norag = []
if os.path.exists(NORAG_PATH):
    with open(NORAG_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["generated_answer"]:
                existing_norag.append(rec)
    with open(NORAG_PATH, "w", encoding="utf-8") as f:
        for rec in existing_norag:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

done_norag_qids = {rec["qid"] for rec in existing_norag}
print(f"Da co san {len(done_norag_qids)} cau tra loi No-RAG tu lan chay truoc")

qids_to_run = common_qids if MAX_QUESTIONS is None else common_qids[:MAX_QUESTIONS]

with open(NORAG_PATH, "a", encoding="utf-8") as f:
    for qid in qids_to_run:
        if qid in done_norag_qids:
            continue
        ex = qid_to_example[qid]
        answer = generate_norag_answer(ex["question"])
        record = {"qid": qid, "question": ex["question"], "gold_answer": ex["answer"], "generated_answer": answer}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        time.sleep(SLEEP_BETWEEN_CALLS)

print("Hoan tat sinh No-RAG.")

## 6. Cập nhật tập câu hỏi dùng để so sánh

Sau bước 5, load lại `no_rag_results.jsonl` và giao thêm với `common_qids` — vì No-RAG cũng có thể chưa chạy hết nếu bị rate-limit giữa chừng.

In [ ]:
norag_by_qid = load_jsonl_by_qid(NORAG_PATH)
final_qids = sorted(set(common_qids) & set(norag_by_qid))
print(f"So cau co du ca 3 he (dung de danh gia va lap bang so sanh): {len(final_qids)}")

## 7. Judge dùng chung: Correctness / Support / Usefulness

`ISSUP_PROMPT`/`ISUSE_PROMPT` giữ nguyên văn từ Notebook 3 để đảm bảo tiêu chí chấm nhất quán giữa các hệ. `CORRECTNESS_PROMPT` là chỉ số mới — so sánh trực tiếp với `answer` gold trong `train.json`, việc mà Notebook 2/3 chưa làm.

In [ ]:
CORRECTNESS_PROMPT = """Ban la bo phan cham diem do chinh xac cua cau tra loi phap luat so voi dap an chuan.

Cau hoi: {question}

Dap an chuan (do con nguoi viet): {gold_answer}

Cau tra loi can cham: {answer}

So sanh y nghia phap ly cot loi cua cau tra loi voi dap an chuan (khong can trung tu ngu, chi can dung noi dung ket luan phap ly).
Chi tra ve JSON dung dinh dang sau, khong giai thich gi them:
{{"label": "CORRECT hoac PARTIALLY_CORRECT hoac INCORRECT", "reason": "ly do ngan gon"}}"""

ISSUP_PROMPT = """Ban la bo phan kiem tra tinh duoc ho tro cua cau tra loi dua tren bang chung.

Cau hoi: {question}

Cau tra loi can kiem tra:
{answer}

Bang chung (cac dieu luat da duoc dung de tra loi):
{evidence_block}

Danh gia cau tra loi co duoc bang chung tren ho tro khong.
Chi tra ve JSON dung dinh dang sau, khong giai thich gi them:
{{"label": "FULLY_SUPPORTED hoac PARTIALLY_SUPPORTED hoac NOT_SUPPORTED", "reason": "ly do ngan gon"}}"""

ISUSE_PROMPT = """Ban la bo phan danh gia do huu ich cua cau tra loi doi voi nguoi hoi.

Cau hoi: {question}

Cau tra loi: {answer}

Danh gia cau tra loi co huu ich, day du, va dung trong tam cau hoi khong.
Chi tra ve JSON dung dinh dang sau, khong giai thich gi them:
{{"label": "USEFUL hoac PARTIALLY_USEFUL hoac NOT_USEFUL", "reason": "ly do ngan gon"}}"""


def judge_correctness(question, answer, gold_answer):
    if not answer:
        return "INCORRECT", "Khong co cau tra loi (rong)"
    prompt = CORRECTNESS_PROMPT.format(question=question, gold_answer=gold_answer, answer=answer)
    result = extract_json(chat(prompt, temperature=0))
    if isinstance(result, dict) and result.get("label") in ("CORRECT", "PARTIALLY_CORRECT", "INCORRECT"):
        return result["label"], result.get("reason", "")
    return "PARTIALLY_CORRECT", "Khong parse duoc JSON, mac dinh trung lap"


def judge_support(question, answer, evidence_chunks):
    if not evidence_chunks:
        return "NOT_APPLICABLE", "Khong co evidence de doi chieu"
    prompt = ISSUP_PROMPT.format(question=question, answer=answer, evidence_block=build_context(evidence_chunks))
    result = extract_json(chat(prompt, temperature=0))
    if isinstance(result, dict) and result.get("label") in ("FULLY_SUPPORTED", "PARTIALLY_SUPPORTED", "NOT_SUPPORTED"):
        return result["label"], result.get("reason", "")
    return "PARTIALLY_SUPPORTED", "Khong parse duoc JSON, mac dinh trung lap"


def judge_usefulness(question, answer):
    prompt = ISUSE_PROMPT.format(question=question, answer=answer)
    result = extract_json(chat(prompt, temperature=0))
    if isinstance(result, dict) and result.get("label") in ("USEFUL", "PARTIALLY_USEFUL", "NOT_USEFUL"):
        return result["label"], result.get("reason", "")
    return "PARTIALLY_USEFUL", "Khong parse duoc JSON, mac dinh trung lap"

## 8. Chạy đánh giá hậu kiểm (checkpoint từng câu, resume-safe)

Mỗi câu tốn 6 lần gọi mới (2 cho No-RAG, 3 cho Standard RAG, 1 cho Self-RAG — vì Self-RAG **tái dùng** `issup_label`/`isuse_label` đã có sẵn từ Notebook 3, không chấm lại).

In [ ]:
EVAL_PATH = os.path.join(ARTIFACT_DIR, "evaluation_details.jsonl")

existing_eval = []
if os.path.exists(EVAL_PATH):
    with open(EVAL_PATH, encoding="utf-8") as f:
        for line in f:
            existing_eval.append(json.loads(line))
done_eval_qids = {r["qid"] for r in existing_eval}
print(f"Da co san danh gia cho {len(done_eval_qids)} cau tu lan chay truoc")

eval_qids_to_run = final_qids if MAX_QUESTIONS is None else final_qids[:MAX_QUESTIONS]

with open(EVAL_PATH, "a", encoding="utf-8") as f:
    for qid in eval_qids_to_run:
        if qid in done_eval_qids:
            continue
        ex = qid_to_example[qid]
        question, gold_answer = ex["question"], ex["answer"]

        norag_answer = norag_by_qid[qid]["generated_answer"]
        standard_rec = standard_by_qid[qid]
        self_rec = self_by_qid[qid]
        standard_answer = standard_rec["generated_answer"]
        self_answer = self_rec["generated_answer"]

        norag_correctness, norag_correctness_reason = judge_correctness(question, norag_answer, gold_answer)
        norag_usefulness, norag_usefulness_reason = judge_usefulness(question, norag_answer)

        standard_evidence = aids_to_chunks(standard_rec["retrieved_aids"])
        standard_correctness, standard_correctness_reason = judge_correctness(question, standard_answer, gold_answer)
        standard_support, standard_support_reason = judge_support(question, standard_answer, standard_evidence)
        standard_usefulness, standard_usefulness_reason = judge_usefulness(question, standard_answer)

        self_correctness, self_correctness_reason = judge_correctness(question, self_answer, gold_answer)

        record = {
            "qid": qid,
            "question": question,
            "gold_answer": gold_answer,
            "gold_relevant_laws": ex["relevant_laws"],

            "norag_answer": norag_answer,
            "norag_correctness": norag_correctness,
            "norag_usefulness": norag_usefulness,

            "standard_answer": standard_answer,
            "standard_retrieved_aids": standard_rec["retrieved_aids"],
            "standard_correctness": standard_correctness,
            "standard_support": standard_support,
            "standard_usefulness": standard_usefulness,

            "self_answer": self_answer,
            "self_retrieve_decision": self_rec["retrieve_decision"],
            "self_retrieved_aids": self_rec["retrieved_aids"],
            "self_relevant_aids": self_rec["relevant_aids"],
            "self_correctness": self_correctness,
            "self_support": self_rec["issup_label"],
            "self_usefulness": self_rec["isuse_label"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        time.sleep(SLEEP_BETWEEN_CALLS)

print("Hoan tat danh gia.")

## 9. Chỉ số retrieval: Standard RAG (top-5 cố định) vs Self-RAG (sau `[ISREL]`)

Tính thuần Python trên `aid`, không tốn API call. So sánh recall **trước** và **sau** lọc ISREL của Self-RAG cho thấy lọc nhiễu có làm mất evidence đúng hay không (kỳ vọng: recall gần như giữ nguyên, precision tăng).

In [ ]:
import pandas as pd


def recall_at_k(retrieved, gold):
    if not gold:
        return None
    return len(set(retrieved) & set(gold)) / len(gold)


def precision_at_k(retrieved, gold):
    if not retrieved:
        return 0.0
    return len(set(retrieved) & set(gold)) / len(retrieved)


def reciprocal_rank(retrieved, gold):
    for rank, aid in enumerate(retrieved, start=1):
        if aid in gold:
            return 1.0 / rank
    return 0.0


eval_df = pd.read_json(EVAL_PATH, lines=True)
eval_df = eval_df[eval_df["qid"].isin(final_qids)].drop_duplicates(subset="qid", keep="last")
print(f"So cau dung de lap bang so sanh: {len(eval_df)}")

retrieval_rows = []
for _, row in eval_df.iterrows():
    gold = set(row["gold_relevant_laws"])
    retrieval_rows.append({
        "qid": row["qid"],
        "standard_recall": recall_at_k(row["standard_retrieved_aids"], gold),
        "standard_precision": precision_at_k(row["standard_retrieved_aids"], gold),
        "standard_mrr": reciprocal_rank(row["standard_retrieved_aids"], gold),
        "self_pre_filter_recall": recall_at_k(row["self_retrieved_aids"], gold) if row["self_retrieved_aids"] else None,
        "self_recall": recall_at_k(row["self_relevant_aids"], gold),
        "self_precision": precision_at_k(row["self_relevant_aids"], gold),
        "self_mrr": reciprocal_rank(row["self_relevant_aids"], gold),
    })
retrieval_df = pd.DataFrame(retrieval_rows)

print("Recall truoc loc ISREL (Self-RAG):", retrieval_df["self_pre_filter_recall"].mean())
print("Recall sau loc ISREL (Self-RAG):  ", retrieval_df["self_recall"].mean())
print("-> Neu 2 so gan bang nhau: loc ISREL khong lam mat evidence dung, chi bo bot nhieu.")

## 10. Bảng so sánh cuối cùng — `final_comparison_table.csv`

`*_rate`: tỷ lệ đạt nhãn tốt nhất tuyệt đối. `*_score`: điểm có trọng số (đầy đủ=1, một phần=0.5, không=0) — sắc thái hơn `*_rate` khi phần lớn câu trả lời chỉ đạt mức "partially".

In [ ]:
def rate(series, positive_labels):
    return series.isin(positive_labels).mean()


def score(series, mapping):
    return series.map(mapping).mean()


CORRECTNESS_SCORE = {"CORRECT": 1.0, "PARTIALLY_CORRECT": 0.5, "INCORRECT": 0.0}
SUPPORT_SCORE = {"FULLY_SUPPORTED": 1.0, "PARTIALLY_SUPPORTED": 0.5, "NOT_SUPPORTED": 0.0}
USEFULNESS_SCORE = {"USEFUL": 1.0, "PARTIALLY_USEFUL": 0.5, "NOT_USEFUL": 0.0}

n = len(eval_df)
summary_rows = [
    {
        "He thong": "No-RAG",
        "Recall": None, "Precision": None, "MRR": None,
        "Correctness_rate": rate(eval_df["norag_correctness"], ["CORRECT"]),
        "Correctness_score": score(eval_df["norag_correctness"], CORRECTNESS_SCORE),
        "Support_rate": None,
        "Usefulness_rate": rate(eval_df["norag_usefulness"], ["USEFUL"]),
        "Usefulness_score": score(eval_df["norag_usefulness"], USEFULNESS_SCORE),
        "So_cau": n,
    },
    {
        "He thong": "Standard RAG",
        "Recall": retrieval_df["standard_recall"].mean(),
        "Precision": retrieval_df["standard_precision"].mean(),
        "MRR": retrieval_df["standard_mrr"].mean(),
        "Correctness_rate": rate(eval_df["standard_correctness"], ["CORRECT"]),
        "Correctness_score": score(eval_df["standard_correctness"], CORRECTNESS_SCORE),
        "Support_rate": rate(eval_df["standard_support"], ["FULLY_SUPPORTED"]),
        "Usefulness_rate": rate(eval_df["standard_usefulness"], ["USEFUL"]),
        "Usefulness_score": score(eval_df["standard_usefulness"], USEFULNESS_SCORE),
        "So_cau": n,
    },
    {
        "He thong": "Self-RAG-inspired",
        "Recall": retrieval_df["self_recall"].mean(),
        "Precision": retrieval_df["self_precision"].mean(),
        "MRR": retrieval_df["self_mrr"].mean(),
        "Correctness_rate": rate(eval_df["self_correctness"], ["CORRECT"]),
        "Correctness_score": score(eval_df["self_correctness"], CORRECTNESS_SCORE),
        "Support_rate": rate(eval_df["self_support"], ["FULLY_SUPPORTED"]),
        "Usefulness_rate": rate(eval_df["self_usefulness"], ["USEFUL"]),
        "Usefulness_score": score(eval_df["self_usefulness"], USEFULNESS_SCORE),
        "So_cau": n,
    },
]
summary_df = pd.DataFrame(summary_rows)

COMPARISON_PATH = os.path.join(ARTIFACT_DIR, "final_comparison_table.csv")
summary_df.to_csv(COMPARISON_PATH, index=False)
print(f"Da luu: {COMPARISON_PATH}")
summary_df

## 11. Case study định tính

Ưu tiên các câu mà Self-RAG đã lọc bớt passage (`self_relevant_aids` ngắn hơn `self_retrieved_aids`) — đúng ví dụ minh họa "ISREL lọc nhiễu" cho phần Discussion của báo cáo.

In [ ]:
filtered_cases = eval_df[eval_df.apply(lambda r: len(r["self_relevant_aids"]) < len(r["self_retrieved_aids"]), axis=1)]
print(f"So cau ISREL thuc su loc bot passage: {len(filtered_cases)}/{len(eval_df)}")

sample_pool = filtered_cases if len(filtered_cases) > 0 else eval_df
for _, row in sample_pool.sample(min(3, len(sample_pool)), random_state=0).iterrows():
    print("\n" + "=" * 80)
    print("qid:", row["qid"])
    print("Cau hoi:", row["question"])
    print("Gold answer:", row["gold_answer"])
    print("\n--- No-RAG ---")
    print("Answer:", row["norag_answer"])
    print("Correctness:", row["norag_correctness"], "| Usefulness:", row["norag_usefulness"])
    print("\n--- Standard RAG ---")
    print("Retrieved aid:", row["standard_retrieved_aids"])
    print("Answer:", row["standard_answer"])
    print("Correctness:", row["standard_correctness"], "| Support:", row["standard_support"], "| Usefulness:", row["standard_usefulness"])
    print("\n--- Self-RAG-inspired ---")
    print("[Retrieve]:", row["self_retrieve_decision"], "| Retrieved aid:", row["self_retrieved_aids"], "| Relevant aid (sau ISREL):", row["self_relevant_aids"])
    print("Answer:", row["self_answer"])
    print("Correctness:", row["self_correctness"], "| Support:", row["self_support"], "| Usefulness:", row["self_usefulness"])

## 12. Bước tiếp theo

- Dùng `final_comparison_table.csv` + case study ở trên trực tiếp cho phần Results/Discussion của báo cáo (khung §28 trong `SELF_RAG_SEMINAR_PROJECT_GUIDELINE.md`).
- Nếu cỡ mẫu (`So_cau` ở bảng trên) còn nhỏ do rate-limit, chạy lại notebook 2/3 để tăng số câu hoàn thành trước khi chạy lại notebook này — mọi thứ đều resume-safe.
- **Phương án (b)** (tùy chọn, làm sau): `05_drill_submission.ipynb` — chạy hệ tốt nhất trên `public_test.json`/`private_test.json`, nộp lên leaderboard VLSP2025 DRiLL.